In [10]:
!pip install -q "datasets==3.5.0" transformers accelerate evaluate sentencepiece sacremoses

In [11]:
import os
import re
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from torch.utils.data import Dataset
from sklearn.metrics import f1_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


In [12]:
LANGUAGES = {
    "hau": "Hausa",
    "yor": "Yoruba",
    "ibo": "Igbo"
}

MODELS = {
    "mbert": "bert-base-multilingual-cased",
    "afriberta": "castorini/afriberta_large",
    "afroxlmr": "Davlan/afro-xlmr-base"
}

LABEL_NAMES = ["negative", "neutral", "positive"]
NUM_LABELS = 3
MAX_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
OUTPUT_DIR = "/kaggle/working/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [13]:
def clean_tweet(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'\bRT\b', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def load_and_preprocess(lang_code):
    print(f"\nLoading {LANGUAGES[lang_code]} ({lang_code})...")
    dataset = load_dataset(
        "shmuhammad/AfriSenti-twitter-sentiment",
        lang_code
    )

    processed = {}
    for split in ["train", "validation", "test"]:
        df = pd.DataFrame(dataset[split])
        df["cleaned_text"] = df["tweet"].apply(clean_tweet)
        df = df[df["cleaned_text"].str.strip() != ""]
        processed[split] = df
        print(f"  {split}: {len(df)} samples")

    return processed

In [14]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, predictions, average="weighted")
    acc = accuracy_score(labels, predictions)
    return {"weighted_f1": f1, "accuracy": acc}

In [15]:
def finetune_model(model_key, lang_code, data):
    model_name = MODELS[model_key]
    lang_name = LANGUAGES[lang_code]

    print(f"\n{'='*60}")
    print(f"Fine-tuning {model_key.upper()} on {lang_name} ({lang_code})")
    print(f"Model: {model_name}")
    print("="*60)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        ignore_mismatched_sizes=True
    )

    train_dataset = SentimentDataset(
        data["train"]["cleaned_text"],
        data["train"]["label"],
        tokenizer,
        MAX_LENGTH
    )
    val_dataset = SentimentDataset(
        data["validation"]["cleaned_text"],
        data["validation"]["label"],
        tokenizer,
        MAX_LENGTH
    )
    test_dataset = SentimentDataset(
        data["test"]["cleaned_text"],
        data["test"]["label"],
        tokenizer,
        MAX_LENGTH
    )

    output_path = f"{OUTPUT_DIR}/{model_key}_{lang_code}"

    training_args = TrainingArguments(
        output_dir=output_path,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="weighted_f1",
        greater_is_better=True,
        logging_steps=50,
        fp16=True,
        report_to="none",
        save_total_limit=1,
        save_only_model=True
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    predictions = trainer.predict(test_dataset)
    preds = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids

    test_f1 = f1_score(labels, preds, average="weighted")
    test_acc = accuracy_score(labels, preds)

    print(f"\nTest Weighted F1: {test_f1:.4f}")
    print(f"Test Accuracy:    {test_acc:.4f}")
    print("\nDetailed Report:")
    print(classification_report(
        labels, preds, target_names=LABEL_NAMES, digits=4
    ))

    del model
    torch.cuda.empty_cache()

    return {
        "model": model_key,
        "lang": lang_code,
        "test_f1": test_f1,
        "test_acc": test_acc,
        "preds": preds.tolist(),
        "labels": labels.tolist()
    }

In [16]:
all_data = {}
for lang_code in LANGUAGES:
    all_data[lang_code] = load_and_preprocess(lang_code)


Loading Hausa (hau)...
  train: 14172 samples
  validation: 2677 samples
  test: 5303 samples

Loading Yoruba (yor)...
  train: 8522 samples
  validation: 2090 samples
  test: 4515 samples

Loading Igbo (ibo)...
  train: 10192 samples
  validation: 1841 samples
  test: 3682 samples


In [17]:
all_results = []

for model_key in MODELS:
    for lang_code in LANGUAGES:
        result = finetune_model(model_key, lang_code, all_data[lang_code])
        all_results.append(result)


Fine-tuning MBERT on Hausa (hau)
Model: bert-base-multilingual-cased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated

Epoch,Training Loss,Validation Loss,Weighted F1,Accuracy
1,1.583228,1.495280,0.655314,0.653717
2,1.261373,1.282719,0.726009,0.727680
3,1.041365,1.286372,0.741585,0.741502
4,0.792016,1.385388,0.736467,0.737393
5,0.653288,1.513927,0.741860,0.742996


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Test Weighted F1: 0.7637
Test Accuracy:    0.7637

Detailed Report:
              precision    recall  f1-score   support

    negative     0.8595    0.7549    0.8038      1118
     neutral     0.7363    0.8199    0.7758      1621
    positive     0.7140    0.6776    0.6953       943

    accuracy                         0.7637      3682
   macro avg     0.7699    0.7508    0.7583      3682
weighted avg     0.7680    0.7637    0.7637      3682



In [18]:
print("\n" + "="*60)
print("FINE-TUNING RESULTS SUMMARY")
print("="*60)
print(f"{'Model':<12} {'Language':<12} {'Test F1':>10} {'Test Acc':>10}")
print("-"*50)

for r in all_results:
    print(
        f"{r['model']:<12}"
        f"{LANGUAGES[r['lang']]:<12}"
        f"{r['test_f1']:>10.4f}"
        f"{r['test_acc']:>10.4f}"
    )

results_df = pd.DataFrame([{
    "model": r["model"],
    "language": LANGUAGES[r["lang"]],
    "lang_code": r["lang"],
    "test_f1": r["test_f1"],
    "test_acc": r["test_acc"]
} for r in all_results])

results_df.to_csv(f"{OUTPUT_DIR}/finetuning_results.csv", index=False)
print(f"\nResults saved to {OUTPUT_DIR}/finetuning_results.csv")


FINE-TUNING RESULTS SUMMARY
Model        Language        Test F1   Test Acc
--------------------------------------------------
mbert       Hausa           0.7337    0.7347
mbert       Yoruba          0.6719    0.6720
mbert       Igbo            0.7618    0.7634
afriberta   Hausa           0.7953    0.7948
afriberta   Yoruba          0.7274    0.7243
afriberta   Igbo            0.7840    0.7849
afroxlmr    Hausa           0.7782    0.7786
afroxlmr    Yoruba          0.6977    0.6937
afroxlmr    Igbo            0.7637    0.7637

Results saved to /kaggle/working/results/finetuning_results.csv


In [19]:
import nltk
from nltk.corpus import wordnet
import random
from transformers import MarianMTModel, MarianTokenizer

nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')


TRANSLATION_MODELS = {
    "hau": ("Helsinki-NLP/opus-mt-ha-en", "Helsinki-NLP/opus-mt-en-ha"),
    "yor": ("Helsinki-NLP/opus-mt-yo-en", "Helsinki-NLP/opus-mt-en-yo"),
    "ibo": ("Helsinki-NLP/opus-mt-ig-en", "Helsinki-NLP/opus-mt-en-ig")
}

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [ ]:
def load_translation_model(model_name):

    try:
        tokenizer = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(model_name).to(device)
        return tokenizer, model
    except Exception as e:
        print(f"  Could not load {model_name}: {e}")
        return None, None


def translate_batch(texts, tokenizer, model, batch_size=16):

    translations = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)
    return translations


def back_translate(texts, lang_code):

    to_en_name, from_en_name = TRANSLATION_MODELS[lang_code]

    print(f"  Loading {lang_code} -> English model...")
    to_en_tok, to_en_model = load_translation_model(to_en_name)
    if to_en_model is None:
        print(f"  Back-translation not available for {lang_code}, skipping.")
        return None

    print(f"  Loading English -> {lang_code} model...")
    from_en_tok, from_en_model = load_translation_model(from_en_name)
    if from_en_model is None:
        print(f"  Back-translation not available for {lang_code}, skipping.")
        return None

    print(f"  Translating {len(texts)} texts to English...")
    english_texts = translate_batch(texts, to_en_tok, to_en_model)

    print(f"  Translating back to {lang_code}...")
    back_translated = translate_batch(english_texts, from_en_tok, from_en_model)

    del to_en_model, from_en_model
    torch.cuda.empty_cache()

    return back_translated

In [ ]:
def get_wordnet_synonyms(word):
 
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ")
            if synonym.lower() != word.lower():
                synonyms.add(synonym)
    return list(synonyms)


def synonym_replacement(text, replacement_rate=0.2):
  
    words = text.split()
    new_words = words.copy()

    # Only target words that look like English 
    eligible_indices = [
        i for i, w in enumerate(words)
        if w.isascii() and w.isalpha() and len(w) > 3
    ]

    n_to_replace = max(1, int(len(eligible_indices) * replacement_rate))
    indices_to_replace = random.sample(
        eligible_indices,
        min(n_to_replace, len(eligible_indices))
    )

    for idx in indices_to_replace:
        word = words[idx]
        synonyms = get_wordnet_synonyms(word)
        if synonyms:
            new_words[idx] = random.choice(synonyms)

    return " ".join(new_words)


def apply_synonym_replacement(texts, replacement_rate=0.2):

    return [synonym_replacement(t, replacement_rate) for t in texts]

In [ ]:
def augment_training_data(lang_code, data):
    """
    Augments the training set for a given language using both
    back-translation and synonym replacement.
    """
    train_df = data["train"].copy()

    print(f"\n{'='*55}")
    print(f"Augmenting {LANGUAGES[lang_code]} ({lang_code}) training data")
    print(f"Original training size: {len(train_df)}")
    print("="*55)

    positive_df = train_df[train_df["label"] == 2].copy()
    print(f"Positive class samples: {len(positive_df)}")

    augmented_frames = [train_df]

    print("\nApplying back-translation to positive class...")
    bt_texts = back_translate(
        positive_df["cleaned_text"].tolist(),
        lang_code
    )

    if bt_texts is not None:
        bt_df = positive_df.copy()
        bt_df["cleaned_text"] = bt_texts
        bt_df["augmentation"] = "back_translation"
        augmented_frames.append(bt_df)
        print(f"  Added {len(bt_df)} back-translated samples")
    else:
        print("  Back-translation skipped, model unavailable")

    # --- Synonym replacement on all classes ---
    print("\nApplying synonym replacement to all training samples...")
    sr_df = train_df.copy()
    sr_df["cleaned_text"] = apply_synonym_replacement(
        train_df["cleaned_text"].tolist(),
        replacement_rate=0.2
    )
    sr_df["augmentation"] = "synonym_replacement"
    augmented_frames.append(sr_df)
    print(f"  Added {len(sr_df)} synonym-replaced samples")

    # Combine original + augmented
    augmented_train = pd.concat(augmented_frames, ignore_index=True)
    augmented_train = augmented_train.dropna(subset=["cleaned_text"])
    augmented_train = augmented_train[
        augmented_train["cleaned_text"].str.strip() != ""
    ]

    print(f"\nFinal augmented training size: {len(augmented_train)}")
    print(f"Increase: +{len(augmented_train) - len(train_df)} samples")

    return augmented_train

In [ ]:
def finetune_augmented(lang_code, augmented_train, data):
    
    model_name = MODELS["afriberta"]

    print(f"\n{'='*55}")
    print(f"Retraining AfriBERTa on augmented {LANGUAGES[lang_code]} data")
    print("="*55)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        ignore_mismatched_sizes=True
    )

    train_dataset = SentimentDataset(
        augmented_train["cleaned_text"],
        augmented_train["label"],
        tokenizer,
        MAX_LENGTH
    )
    val_dataset = SentimentDataset(
        data["validation"]["cleaned_text"],
        data["validation"]["label"],
        tokenizer,
        MAX_LENGTH
    )
    test_dataset = SentimentDataset(
        data["test"]["cleaned_text"],
        data["test"]["label"],
        tokenizer,
        MAX_LENGTH
    )

    output_path = f"{OUTPUT_DIR}/afriberta_augmented_{lang_code}"

    training_args = TrainingArguments(
        output_dir=output_path,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="weighted_f1",
        greater_is_better=True,
        logging_steps=50,
        fp16=True,
        report_to="none",
        save_total_limit=1,
        save_only_model=True
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    predictions = trainer.predict(test_dataset)
    preds = np.argmax(predictions.predictions, axis=-1)
    labels = predictions.label_ids

    test_f1 = f1_score(labels, preds, average="weighted")
    test_acc = accuracy_score(labels, preds)

    print(f"\nAugmented Test Weighted F1: {test_f1:.4f}")
    print(f"Augmented Test Accuracy:    {test_acc:.4f}")
    print("\nDetailed Report:")
    print(classification_report(
        labels, preds, target_names=LABEL_NAMES, digits=4
    ))

    del model
    torch.cuda.empty_cache()

    return {
        "lang": lang_code,
        "augmented_f1": test_f1,
        "augmented_acc": test_acc,
        "preds": preds.tolist(),
        "labels": labels.tolist()
    }

In [ ]:
augmented_results = []

for lang_code in LANGUAGES:
    augmented_train = augment_training_data(lang_code, all_data[lang_code])
    result = finetune_augmented(lang_code, augmented_train, all_data[lang_code])
    augmented_results.append(result)


Augmenting Hausa (hau) training data
Original training size: 14172
Positive class samples: 4573

Applying back-translation to positive class...
  Loading hau -> English model...


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/783k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Augmented Test Weighted F1: 0.7872
Augmented Test Accuracy:    0.7879

Detailed Report:
              precision    recall  f1-score   support

    negative     0.8585    0.7818    0.8184      1118
     neutral     0.7449    0.8519    0.7948      1621
    positive     0.7975    0.6850    0.7370       943

    accuracy                         0.7879      3682
   macro avg     0.8003    0.7729    0.7834      3682
weighted avg     0.7929    0.7879    0.7872      3682



In [ ]:
original_afriberta = {
    "hau": 0.7953,
    "yor": 0.7274,
    "ibo": 0.7840
}

print("\n" + "="*60)
print("AUGMENTATION RESULTS: AfriBERTa Original vs Augmented")
print("="*60)
print(f"{'Language':<12} {'Original F1':>14} {'Augmented F1':>14} {'Change':>10}")
print("-"*55)

for r in augmented_results:
    lang = r["lang"]
    orig = original_afriberta[lang]
    aug = r["augmented_f1"]
    change = aug - orig
    direction = "+" if change >= 0 else ""
    print(
        f"{LANGUAGES[lang]:<12}"
        f"{orig:>14.4f}"
        f"{aug:>14.4f}"
        f"{direction}{change:>9.4f}"
    )

aug_df = pd.DataFrame([{
    "language": LANGUAGES[r["lang"]],
    "lang_code": r["lang"],
    "original_f1": original_afriberta[r["lang"]],
    "augmented_f1": r["augmented_f1"],
    "change": r["augmented_f1"] - original_afriberta[r["lang"]]
} for r in augmented_results])

aug_df.to_csv(f"{OUTPUT_DIR}/augmentation_results.csv", index=False)
print(f"\nSaved to {OUTPUT_DIR}/augmentation_results.csv")


AUGMENTATION RESULTS: AfriBERTa Original vs Augmented
Language        Original F1   Augmented F1     Change
-------------------------------------------------------
Hausa               0.7953        0.7683  -0.0270
Yoruba              0.7274        0.7223  -0.0051
Igbo                0.7840        0.7872+   0.0032

Saved to /kaggle/working/results/augmentation_results.csv


In [ ]:
def retrain_for_error_analysis(lang_code):

    model_name = MODELS["afriberta"]
    data = all_data[lang_code]

    print(f"Retraining AfriBERTa on {LANGUAGES[lang_code]} for error analysis...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        ignore_mismatched_sizes=True
    )

    train_dataset = SentimentDataset(
        data["train"]["cleaned_text"],
        data["train"]["label"],
        tokenizer,
        MAX_LENGTH
    )
    val_dataset = SentimentDataset(
        data["validation"]["cleaned_text"],
        data["validation"]["label"],
        tokenizer,
        MAX_LENGTH
    )

    output_path = f"{OUTPUT_DIR}/afriberta_{lang_code}_error_analysis"

    training_args = TrainingArguments(
        output_dir=output_path,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="weighted_f1",
        greater_is_better=True,
        logging_steps=50,
        fp16=True,
        report_to="none",
        save_total_limit=1,
        save_only_model=True
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    return model, tokenizer, data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os

os.makedirs(f"{OUTPUT_DIR}/plots", exist_ok=True)

def plot_confusion_matrix(preds, labels, title, save_path):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=LABEL_NAMES,
        yticklabels=LABEL_NAMES
    )
    plt.title(title)
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f"Saved: {save_path}")


# Plot confusion matrices 
for r in all_results:
    model_key = r["model"]
    lang_code = r["lang"]
    title = f"{model_key.upper()} - {LANGUAGES[lang_code]} Confusion Matrix"
    save_path = f"{OUTPUT_DIR}/plots/{model_key}_{lang_code}_confusion_matrix.png"
    plot_confusion_matrix(r["preds"], r["labels"], title, save_path)

print("\nAll confusion matrices saved.")

Saved: /kaggle/working/results/plots/mbert_hau_confusion_matrix.png
Saved: /kaggle/working/results/plots/mbert_yor_confusion_matrix.png
Saved: /kaggle/working/results/plots/mbert_ibo_confusion_matrix.png
Saved: /kaggle/working/results/plots/afriberta_hau_confusion_matrix.png
Saved: /kaggle/working/results/plots/afriberta_yor_confusion_matrix.png
Saved: /kaggle/working/results/plots/afriberta_ibo_confusion_matrix.png
Saved: /kaggle/working/results/plots/afroxlmr_hau_confusion_matrix.png
Saved: /kaggle/working/results/plots/afroxlmr_yor_confusion_matrix.png
Saved: /kaggle/working/results/plots/afroxlmr_ibo_confusion_matrix.png

All confusion matrices saved.


In [ ]:
def analyse_misclassifications(lang_code, preds, labels, n_examples=10):

    test_df = all_data[lang_code]["test"].copy()
    test_df = test_df.reset_index(drop=True)

    pred_series = pd.Series(preds[:len(test_df)])
    label_series = pd.Series(labels[:len(test_df)])

    test_df["predicted"] = pred_series
    test_df["true_label"] = label_series
    test_df["correct"] = test_df["predicted"] == test_df["true_label"]

    label_map = {0: "negative", 1: "neutral", 2: "positive"}
    test_df["predicted_name"] = test_df["predicted"].map(label_map)
    test_df["true_name"] = test_df["true_label"].map(label_map)

    misclassified = test_df[~test_df["correct"]]

    print(f"\n{'='*55}")
    print(f"Misclassification Analysis - {LANGUAGES[lang_code]}")
    print(f"{'='*55}")
    print(f"Total test samples:     {len(test_df)}")
    print(f"Misclassified:          {len(misclassified)}")
    print(f"Accuracy:               {test_df['correct'].mean():.4f}")

    print(f"\nError breakdown by type:")
    error_counts = misclassified.groupby(
        ["true_name", "predicted_name"]
    ).size().reset_index(name="count")
    error_counts = error_counts.sort_values("count", ascending=False)
    print(error_counts.to_string(index=False))

    print(f"\nSample misclassified tweets (true → predicted):")
    print("-" * 55)

    for _, row in misclassified.head(n_examples).iterrows():
        print(f"True: {row['true_name']:<10} Predicted: {row['predicted_name']}")
        print(f"Tweet: {row['tweet'][:120]}")
        print()

    return misclassified, error_counts


#misclassification analysis for AfriBERTa on all languages
afriberta_results = [r for r in all_results if r["model"] == "afriberta"]

all_error_analysis = {}
for r in afriberta_results:
    lang_code = r["lang"]
    misclassified, error_counts = analyse_misclassifications(
        lang_code,
        r["preds"],
        r["labels"]
    )
    all_error_analysis[lang_code] = {
        "misclassified": misclassified,
        "error_counts": error_counts
    }


Misclassification Analysis - Hausa
Total test samples:     5303
Misclassified:          1088
Accuracy:               0.7948

Error breakdown by type:
true_name predicted_name  count
  neutral       positive    310
 positive        neutral    297
 negative        neutral    168
  neutral       negative    114
 positive       negative    103
 negative       positive     96

Sample misclassified tweets (true → predicted):
-------------------------------------------------------
True: negative   Predicted: positive
Tweet: wannan shine gaskiyar lamari domin siyasar nigeria sai mai hankali zai zabi atiku wllh

True: negative   Predicted: neutral
Tweet: buqatan maji haji sallah

True: negative   Predicted: positive
Tweet: wahya fa sai heqi kike qarawa yasin

True: negative   Predicted: neutral
Tweet: ashe akwae mata masu basira bayan na girki

True: negative   Predicted: neutral
Tweet: rahama yan mata

True: negative   Predicted: positive
Tweet: allah y vashii saa sbd wnn duk sharrin maqiiyan

In [ ]:
from lime.lime_text import LimeTextExplainer
import numpy as np

def run_lime_analysis(lang_code, n_samples=10):
 
    print(f"\nRetraining AfriBERTa on {LANGUAGES[lang_code]} for LIME analysis...")
    model, tokenizer, data = retrain_for_error_analysis(lang_code)
    model.eval()

    #misclassified examples from earlier analysis
    misclassified = all_error_analysis[lang_code]["misclassified"]
    sample = misclassified.head(n_samples)

    def predict_proba(texts):

        inputs = tokenizer(
            list(texts),
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)

        return probs.cpu().numpy()

    explainer = LimeTextExplainer(class_names=LABEL_NAMES)

    lime_results = []
    print(f"\nGenerating LIME explanations for {n_samples} misclassified examples...")

    for idx, (_, row) in enumerate(sample.iterrows()):
        text = row["cleaned_text"]
        true_label = int(row["true_label"])
        pred_label = int(row["predicted"])

        try:
            exp = explainer.explain_instance(
                text,
                predict_proba,
                num_features=8,
                num_samples=100,
                labels=[pred_label]
            )

            top_features = exp.as_list(label=pred_label)

            print(f"\nExample {idx + 1}:")
            print(f"  True: {LABEL_NAMES[true_label]} | Predicted: {LABEL_NAMES[pred_label]}")
            print(f"  Tweet: {text[:100]}")
            print(f"  Top features driving prediction:")
            for word, weight in top_features[:5]:
                direction = "positive" if weight > 0 else "negative"
                print(f"    '{word}': {weight:.4f} ({direction} influence)")

            # Save the explanation as an image
            fig = exp.as_pyplot_figure(label=pred_label)
            save_path = (
                f"{OUTPUT_DIR}/plots/lime_{lang_code}_example_{idx+1}.png"
            )
            fig.savefig(save_path, dpi=150, bbox_inches="tight")
            plt.close()

            lime_results.append({
                "example": idx + 1,
                "true_label": LABEL_NAMES[true_label],
                "predicted_label": LABEL_NAMES[pred_label],
                "tweet": text,
                "top_features": top_features[:5]
            })

        except Exception as e:
            print(f"  LIME failed for example {idx+1}: {e}")
            continue

    del model
    torch.cuda.empty_cache()

    return lime_results


lime_results = run_lime_analysis("hau", n_samples=10)


Retraining AfriBERTa on Hausa for LIME analysis...
Retraining AfriBERTa on Hausa for error analysis...


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: castorini/afriberta_large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Weighted F1,Accuracy
1,1.216960,1.085081,0.776919,0.774748
2,0.854390,1.038573,0.792327,0.791931
3,0.587308,1.211459,0.787579,0.788196
4,0.314800,1.464602,0.792302,0.791931


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Generating LIME explanations for 10 misclassified examples...

Example 1:
  True: negative | Predicted: positive
  Tweet: wannan shine gaskiyar lamari domin siyasar nigeria sai mai hankali zai zabi atiku wllh
  Top features driving prediction:
    'wllh': 0.2423 (positive influence)
    'hankali': 0.2178 (positive influence)
    'zabi': -0.2144 (negative influence)
    'siyasar': 0.1080 (positive influence)
    'atiku': 0.0864 (positive influence)

Example 2:
  True: negative | Predicted: neutral
  Tweet: buqatan maji haji sallah
  Top features driving prediction:
    'haji': -0.0658 (negative influence)
    'maji': -0.0353 (negative influence)
    'sallah': -0.0230 (negative influence)
    'buqatan': -0.0029 (negative influence)

Example 3:
  True: negative | Predicted: positive
  Tweet: wahya fa sai heqi kike qarawa yasin
  Top features driving prediction:
    'yasin': 0.2098 (positive influence)
    'kike': 0.1603 (positive influence)
    'heqi': 0.0951 (positive influence)
    'qa

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/results_export",
    "zip",
    "/kaggle/working/results"
)

Done. Download results_export.zip from the output panel.
